# RSA step 5 on GPU -- straight from the result zips on Drive

Builds the group null for one or more RSA models: for each of `REPS_GROUP` group
permutations it draws **one** of every participant's step-4 permutation maps and
averages them voxelwise (`rsa_utils.calculate_group_model_similarity_map_rnd`).

**Input** is the `result_<model>_<specie>-sub-NN.zip` files a previous Colab
step-1/2/4 run already wrote to Drive. Nothing else -- no `pkg_group_*.zip`, no
mask, no network disk:

* every parameter (dataset, GLM model, radius, `dis_method`, `rsa_method`,
  `mah_fold`, the per-run layout, who has which permutations) is read out of the
  arcnames inside the zips;
* step 5 calls `nifti_mean` **without** a mask, so it is a plain voxelwise mean
  and no mask is needed -- see the `gpu_step5.py` docstring.

Every map is read exactly once, scatter-added into a `(REPS_GROUP, n_voxels)`
accumulator on the GPU. The CPU step re-reads each map ~`REPS_GROUP*units/maps`
times (10x at EmoC human scale), so this is the win -- the run is bounded by
reading Drive, not by arithmetic.

## Two things worth knowing before you run it

1. **Step 5 alone is bulky output.** 1000 group mean maps on the EmoC human grid
   are roughly **450 MB per model** to write to Drive and download again. Those
   maps are inputs to steps 6 and 7 and to nothing else. If you also want 6 and 7,
   use `colab_rsa_group.ipynb` instead with `WRITE_GROUP_MEANS = False`: it does
   3/5/6/7 in the same pass and brings back a few megabytes.
2. **The availability gate needs a denominator.** The CPU compares against the
   config's participant list, which is not on Drive. `EXPECTED_PARTICIPANTS`
   below says what to compare against instead.

## Steps
1. Put `gpu_step5.py` in `CODE_DIR` on Drive (see the copy command in cell 3).
2. Set `RESULTS_DIR`, `OUT_DIR`, `SPECIE`, `MODELS`.
3. Run all cells. One `result_step5_<model>_<specie>.zip` per model appears in
   `OUT_DIR`; re-running skips models already done.
4. Back on the workstation: `tools/unpack_results.py <downloads>` merges them onto
   the pipeline disk, then run steps 6-10 as usual.

In [ ]:
# 1. Check the GPU and install nibabel (torch is preinstalled on Colab).
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
!pip -q install nibabel

In [ ]:
# 2. Mount Google Drive.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. EDIT THESE.
#
# gpu_step5.py is a single self-contained file. Put it next to this notebook on
# Drive once, from the workstation (Anaconda Prompt):
#
#   copy \github\dog_brain_toolkit\tools\colab_gpu\gpu_step5.py "G:\My Drive\rsa_colab\"
#
CODE_DIR    = '/content/drive/MyDrive/rsa_colab'            # where gpu_step5.py lives
RESULTS_DIR = '/content/drive/MyDrive/rsa_colab/results'    # the result_<model>_<specie>-sub-NN.zip
OUT_DIR     = '/content/drive/MyDrive/rsa_colab/results'    # where result_step5_*.zip go

SPECIE      = 'H'                        # 'H', 'D', or None for both
MODELS      = ['action_tendency__all']   # None = every model in RESULTS_DIR
REPS_GROUP  = 1000                       # group permutations (searchlight.py --reps_group)

# Availability gate, the equivalent of min_percentage_available in the CPU step.
# EXPECTED_PARTICIPANTS is the denominator:
#   'auto'  -- every <SPECIE>-sub-NN that appears in ANY result zip in RESULTS_DIR,
#              so a participant who finished other models but not this one still
#              counts as expected. This is the useful setting.
#   <int>   -- the number you know the config's participant list has.
#   'found' -- the participants found, i.e. always 100%: disables the check.
MIN_PERCENTAGE_AVAILABLE = 0.5
EXPECTED_PARTICIPANTS    = 'auto'

WORKERS = 8      # threads for writing the output niftis
FORCE   = False  # True: recompute even if result_step5_*.zip already exists

In [ ]:
# 4. Load the module and see what the zips say -- no compute yet.
import sys
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
import gpu_step5

specs = gpu_step5.scan_results(RESULTS_DIR, specie=SPECIE, models=MODELS)
for key in sorted(specs):
    spec = specs[key]
    expected, how = gpu_step5.resolve_expected_participants(spec, EXPECTED_PARTICIPANTS)
    pct = 100 * len(spec.participants) / expected
    verdict = 'RUNS' if pct >= 100 * MIN_PERCENTAGE_AVAILABLE else 'SKIPPED'
    print(f"{spec.describe()}\n    {len(spec.participants)}/{expected} participants "
          f"({pct:.1f}%) vs gate {100 * MIN_PERCENTAGE_AVAILABLE:.0f}% -> {verdict}"
          f"   [denominator: {how}]")
    print(f"    output: ~{REPS_GROUP} maps -> result_step5_{spec.rsa_model}_{spec.specie}.zip")

In [ ]:
# 5. Run step 5. Resumable: skips models whose result_step5_*.zip already exists.
written = gpu_step5.run_step5_all(
    RESULTS_DIR, OUT_DIR, specie=SPECIE, models=MODELS,
    reps_group=REPS_GROUP, work_root='/content/step5_work',
    min_percentage_available=MIN_PERCENTAGE_AVAILABLE,
    expected_participants=EXPECTED_PARTICIPANTS,
    workers=WORKERS, force=FORCE, verbose=True)

print('\nnew step-5 result zips:')
for w in written:
    print(' ', w)

## Back on the workstation

Merge the zips onto the pipeline disk and carry on from step 6 (Anaconda Prompt):

```
python \github\dog_brain_toolkit\tools\unpack_results.py \path\to\downloads --dry-run
python \github\dog_brain_toolkit\tools\unpack_results.py \path\to\downloads
python \github\dog_brain_toolkit\searchlight.py --dataset EmoC --model basic-block --specie H --rsa_model action_tendency__all --steps_to_run 6 7 8 9 10
```

The arcnames inside `result_step5_*.zip` are the same pipeline-relative paths the
CPU step writes, so step 6 finds the maps exactly where it expects them.